# Inspect a causal attention block

Prerequisites: Matrix multiplication, dot products, softmax and sequence order.

Attention forms query, key and value vectors with distinct linear maps. Scores QKᵀ/√d compare each query with keys. Row-wise softmax creates weights that combine the value vectors. In a causal sequence model, positions after the query position receive a masked score before softmax; their weights must be zero.

The companion processes three two-dimensional tokens with one attention head, a residual connection and a small feed-forward transformation. Weights are fixed and inspectable. It is a tiny transformer-style block rather than a trained language model: there is no tokenizer, corpus, optimizer, vocabulary projection, positional encoding or generation service. Those omissions matter when moving from this mechanism to a usable predictor.

A strong causality check changes only the last token. Earlier outputs must stay exactly unchanged in this implementation. Inspecting a plausible-looking attention heatmap would not establish that property. Normalized attention weights describe this internal mixing operation; they are not automatically calibrated explanations of a model's reasoning or factual reliability.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Scores | QKᵀ/√d | Token-to-token compatibility |
| Mask | Future scores = −∞ | Causal information boundary |
| Mixing | softmax(scores)V | Weighted value vectors |

## Predict

The first token can attend only to itself, so its weight row is (1,0,0). Every row sums to one. Increasing the last token by 20 must leave the first two outputs unchanged.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import ai_models as m

In [ ]:
result=m.attention()
assert result['attention'][0]==[1.0,0.0,0.0]
assert max(abs(x-1) for x in result['row_sums'])<1e-12
assert result['earlier_token_change']==0
print(result)

## Perturb and explain

Remove the mask in a copied function and repeat the future-token perturbation. Explain which prediction task would then leak future information.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

Where should the causal mask act?

1. After converting weights into prose
2. On attention scores before softmax
3. Only on the final displayed answer

Answer and explanations: 1: No prose exists in this block. 2: Masking scores prevents future values from receiving weight. 3: A display change cannot undo information used in computation.

## Primary evidence

- [Vaswani et al. — Attention Is All You Need, §3](https://arxiv.org/abs/1706.03762)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.